In [1]:
import os
os.chdir("/home/mdarif/MLOPS/Wine-Quality-Prediction")
%pwd

'/home/mdarif/MLOPS/Wine-Quality-Prediction'

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_file: Path
    database_table: str
    dataset_url: str

In [3]:
from wine_quality_prediction.constants import *
from wine_quality_prediction.utils.common import read_yaml, create_directories

In [4]:
CONFIG_FILE_PATH

PosixPath('config/config.yaml')

In [ ]:
class ConfigurationManager:

    def __init__(
            self, 
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH,
            schema_filepath = SCHEMA_FILE_PATH):

            self.config = read_yaml(config_filepath)
            self.params = read_yaml(params_filepath)
            self.schema = read_yaml(schema_filepath)

            create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self):
          config = self.config.data_ingestion

          create_directories([config.root_dir])

          data_ingestion_config = DataIngestionConfig(
                root_dir=Path(config.root_dir),
                source_file=Path(config.source_file),
                database_table=config.database_table,
                dataset_url=config.dataset_url
          )
          return data_ingestion_config


In [6]:
import os 
import urllib.request
import pandas as pd
from wine_quality_prediction.database import DatabaseOperations
from wine_quality_prediction.logger import get_logger, log_exceptions
from wine_quality_prediction.utils.common import get_size

logger = get_logger("data_ingestion", "data_ingestion.log")

In [7]:
class DataIngestion:
    """Handles data loading from CSV and storing raw data into database. """

    def __init__(self, config: DataIngestionConfig):
        """Initialize with ingestion config."""
        self.config = config

    @log_exceptions
    def download_data(self):
        """Download dataset from Github raw URL."""
        if not os.path.exists(self.config.source_file):
            logger.info(f"Downloading dataset from: '{self.config.dataset_url}'")

            urllib.request.urlretrieve(
                self.config.dataset_url,
                self.config.source_file
            )
            logger.info(f"Dataset downloaded successfully to: '{self.config.source_file}'")
            logger.info(f"File size: '{get_size(self.config.source_file)}'")

    @log_exceptions
    def load_csv(self) -> pd.DataFrame:
        """Load raw data from CSV file."""
        file_path = self.config.source_file

        logger.info(f"Reading file: '{file_path}'")
        logger.info(f"File size: '{get_size(file_path)}'")

        df = pd.read_csv(file_path)

        logger.info(f"CSV loaded successfully with shape: '{df.shape}'")
        logger.info(f"Columns found: {df.columns.tolist()}")
        return df

    @log_exceptions
    def store_in_database(self, df: pd.DataFrame):
        """Store raw dataframe into database table."""
        db = DatabaseOperations()

        try:
            logger.info("Creating table if not exists")
            db.create_table()

            existing = db.fetch_data()
            if len(existing) > 0:
                logger.info("Data already in database skipping insert")
                return 

            logger.info("Inserting raw data into database")
            db.insert_dataframe(df)

            logger.info("Data inserted successfully.")
        finally:
            db.close_connection()

    @log_exceptions
    def run(self):
        """Execute full data ingestion pipeline."""
        self.download_data()
        df = self.load_csv()
        self.store_in_database(df)
        logger.info("Data ingestion completed successfully")

In [8]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.run()
except Exception as e:
    raise e

2026-03-25 09:47:27,686 - common - INFO - common.py:24 - YAML file config/config.yaml loaded successfully
2026-03-25 09:47:27,689 - common - INFO - common.py:24 - YAML file params.yaml loaded successfully
2026-03-25 09:47:27,697 - common - INFO - common.py:24 - YAML file schema.yaml loaded successfully
2026-03-25 09:47:27,700 - common - INFO - common.py:36 - Created directory at: 'artifacts'
2026-03-25 09:47:27,703 - common - INFO - common.py:36 - Created directory at: 'artifacts/data_ingestion'
2026-03-25 09:47:27,705 - data_ingestion - INFO - 1335477075.py:26 - Reading file: 'artifacts/data_ingestion/WineQT.csv'
2026-03-25 09:47:27,707 - data_ingestion - INFO - 1335477075.py:27 - File size: '~76 KB'
2026-03-25 09:47:27,723 - data_ingestion - INFO - 1335477075.py:31 - CSV loaded successfully with shape: '(1143, 13)'
2026-03-25 09:47:27,724 - data_ingestion - INFO - 1335477075.py:32 - Columns found: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'fr